In [4]:
# from huggingface_hub import snapshot_download

# snapshot_download(
#     repo_id="malaysia-ai/Emilia-YODAS-Voice-Conversion", 
#     repo_type="dataset", local_dir="./", allow_patterns="Emilia-YODAS_permutate-*.zip")

In [12]:
from glob import glob
import json
from multiprocess import Pool
import itertools

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [9]:
files = glob('Emilia-YODAS_permutate/*/*')
len(files)

331717

In [17]:
from tqdm import tqdm

def loop(files):
    files, _ = files
    data = {}
    for f in tqdm(files):
        with open(f) as fopen:
            d = json.load(fopen)
        for r in d:
            if r['reference_audio'] not in data:
                data[r['reference_audio']] = r['reference_text']
    return [data]

In [21]:
data = multiprocessing(files, loop, cores = 30)

100%|██████████| 11057/11057 [01:32<00:00, 119.99it/s]


In [22]:
merged = {}
for i in range(len(data)):
    merged.update(data[i])

In [24]:
len(merged)

11365354

In [26]:
rows = []
for k, v in merged.items():
    rows.append({
        'audio_filename': k,
        'text': v,
    })
rows[:10]

[{'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000038.mp3',
  'text': '而更重要的事情是今年瑞赤人影年,也就是我们陈部长的破关之年。这种破关之年,诸事不利,行事倒行逆施。虽然他本人可能没有这个心态,也没有这个想法,但是他当他去实行他的业务的时候,就是会产生这样的结果。'},
 {'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000019.mp3',
  'text': '而像陳部长这样的人呢,他当医生,那是比较糟蹋的。因为这个命格,可以说是标准的一代名将的格局。如果他生在过去的封建时代,或是当年他没有当医生,直接去报考军校,那么他的存在,那会成为让他的敌人文风尚胆的大将军。'},
 {'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000022.mp3',
  'text': '他眼里就是尽力的去打,而且几乎就只有打胜仗这个选项。所以,无论用任何的手段,哪怕牺牲了再多的士兵的生命,甚至这些士兵不是别人,就是他自己的父老兄弟,甚至是他的侄女,只要不伤害到他自己,都是可以牺牲的棋子。'},
 {'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000004.mp3',
  'text': '基本上,陳部長在我們陳水扁前總統第二任的時候,才第一次的被任命為,當年還是被稱作衛生署的副署長。這是他第一次的公職。在陳水扁總統卸任之後,他就回歸本業,繼續當他的專業牙醫師。'},
 {'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000011.mp3',
  'text': '这就像什么呢?这就像我们现任的台北市市长柯文哲市长。他现在是第二任,而且做得还蛮不错的。但是他在第一任的时候啊,他是什么都想做,但是什么都做不好。'},
 {'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000031.mp3',
  'text': '而这个是不是他想不想做好呢?也不是。而是他没有这个经验跟他能力可以做好,因为他累积的这个经验不够多。'},
 

In [27]:
from datasets import Dataset

dataset = Dataset.from_list(rows)
dataset[0]

{'audio_filename': 'Emilia-YODAS/ZH/ZH_rDxEtTgGrzQ_W000038.mp3',
 'text': '而更重要的事情是今年瑞赤人影年,也就是我们陈部长的破关之年。这种破关之年,诸事不利,行事倒行逆施。虽然他本人可能没有这个心态,也没有这个想法,但是他当他去实行他的业务的时候,就是会产生这样的结果。'}

In [29]:
# dataset.push_to_hub('malaysia-ai/Emilia-YODAS-Voice-Conversion', 'audio_text')